# V2 science sanity check
Three independent checks of what LunaNav computes (board task V2; results recorded in docs/decisions.md):
1. **DTE %** vs the LRO LOLA Average Earth Visibility map (Mazarico et al. 2011, Trek `AVGVISIB_85S_060M_201608_EARTH_SP`,
   % of Earth's disc visible, averaged hourly over 18.6 years).
2. **Earth elevation range** vs the libration envelope (sub-Earth latitude/longitude from Horizons over 18.6 years).
3. **Terrain horizon** vs known terrain (Shackleton's floor seen from its rim; what sets each site's highest horizon).

Inputs (all cached in data/raw/): `uv run python -m data.terrain`, `data.sites`, `trek.download_raw(...)`,
`horizons.track('earth', '2027-01-01 00:00', '2045-08-01 00:00', '3h')`.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import rasterio
import rasterio.windows
from pyproj import Transformer

from data import horizons, lola, pgda, terrain
from data.geometry import topocentric
from data.horizons import SITES

SITE_IDS = ["site01_connecting_ridge", "site04_shackleton_rim"]
EARTH_RADIUS_KM = 6371.0
TO_STEREO = Transformer.from_crs("+proj=longlat +R=1737400 +no_defs", pgda.POLAR_STEREO, always_xy=True)

# What the app reads: the committed site files (data/raw/ copies can be stale after a site moves)
site_json = {s: json.loads((ROOT / "public/data/sites" / f"{s}.json").read_text(encoding="utf-8")) for s in SITE_IDS}
hz = {s: site_json[s]["horizons"][0] for s in SITE_IDS}
win = {s: pd.DataFrame({"earth_az_deg": site_json[s]["earth"]["azimuthDeg"], "earth_el_deg": site_json[s]["earth"]["elevationDeg"]})
       for s in SITE_IDS}
ground_m = {s: terrain.site_height_m(s) for s in SITE_IDS}
ANTENNA_M = terrain.OBSERVER_HEIGHT_M


def horizon_at(site: str, az_deg: np.ndarray) -> np.ndarray:
    """Linear interpolation with wrap, same as visibility.ts horizonElevationDeg."""
    h = np.asarray(hz[site]["elevationDeg"], float)
    x = (np.asarray(az_deg) % 360) / hz[site]["azimuthStepDeg"]
    k = np.floor(x).astype(int)
    t = x - k
    return h[k % len(h)] * (1 - t) + h[(k + 1) % len(h)] * t


def disc_fraction(el_deg, horizon_deg, radius_deg):
    """Same circular-segment formula as visibility.ts sunlitFraction."""
    d = np.clip((el_deg - horizon_deg) / radius_deg, -1, 1)
    return 1 - (np.arccos(d) - d * np.sqrt(1 - d * d)) / np.pi

## 1. DTE % vs the Mazarico Earth-visibility map
Our number over the same kind of span (18.6 years, 3 h steps) with our horizon, and over the app's 3-month window.

In [2]:
long = horizons.track("earth", "2027-01-01 00:00", "2045-08-01 00:00", "3h")
lat, lon = np.radians(long.lat_deg.to_numpy()), np.radians(long.lon_deg.to_numpy())
r = long.range_km.to_numpy()
earth_km = np.stack([r * np.cos(lat) * np.cos(lon), r * np.cos(lat) * np.sin(lon), r * np.sin(lat)], -1)
print(f"18.6-year track: {len(long)} samples, {long.time_utc.iloc[0]} → {long.time_utc.iloc[-1]}")

visib = next((ROOT / "data/raw/trek/raw_files").glob("AVGVISIB_85S_060M_201608_EARTH_SP.tif"))
rows = []
with rasterio.open(visib) as ds:
    z = ds.read(1).astype(float)
    z[z < -1e30] = np.nan
    for s in SITE_IDS:
        la, lo = SITES[s]
        x, y = TO_STEREO.transform(lo, la)
        rr, cc = ds.index(x, y)
        box = z[rr - 2: rr + 3, cc - 2: cc + 3]  # 5×5 px = 300 m box: how much the map moves near the site
        az, el, rng = topocentric(la, lo, ground_m[s] + ANTENNA_M, earth_km)
        hor = horizon_at(s, az)
        frac = disc_fraction(el, hor, np.degrees(np.arcsin(EARTH_RADIUS_KM / rng)))
        ser = win[s]
        w_center = (ser.earth_el_deg > horizon_at(s, ser.earth_az_deg)).mean() * 100
        rows.append({
            "site": s, "map_pct": z[rr, cc] * 100, "map_5x5_min": np.nanmin(box) * 100, "map_5x5_max": np.nanmax(box) * 100,
            "ours_18.6y_disc_pct": frac.mean() * 100, "ours_18.6y_center_pct": (el > hor).mean() * 100,
            "ours_window_center_pct": w_center,
        })
dte = pd.DataFrame(rows).set_index("site").round(1)
dte

18.6-year track: 54297 samples, 2027-01-01 00:00:00+00:00 → 2045-08-01 00:00:00+00:00


,map_pct,map_5x5_min,map_5x5_max,ours_18.6y_disc_pct,ours_18.6y_center_pct,ours_window_center_pct
site,,,,,,
site01_connecting_ridge,58.0,40.6,58.0,53.3,53.3,46.9
site04_shackleton_rim,56.7,15.7,56.7,52.0,52.0,45.3


## 2. Earth elevation vs libration
Sub-Earth point range over 18.6 years; Earth's smooth elevation range at each site over 18.6 years and over the window;
and the window series (Horizons observer table) vs our own topocentric math from the Moon-fixed track.

In [3]:
print(f"sub-Earth lat {long.lat_deg.min():.2f}..{long.lat_deg.max():.2f}°, "
      f"lon {((long.lon_deg + 180) % 360 - 180).min():.2f}..{((long.lon_deg + 180) % 360 - 180).max():.2f}°")
wtrack = horizons.earth_moon_fixed()
lib = []
for s in SITE_IDS:
    la, lo = SITES[s]
    _, el, _ = topocentric(la, lo, ground_m[s] + ANTENNA_M, earth_km)
    ser = win[s]
    _, el_w, _ = topocentric(la, lo, ground_m[s] + ANTENNA_M, wtrack[["x_km", "y_km", "z_km"]].to_numpy())
    resid = ser.earth_el_deg.to_numpy() - el_w
    lib.append({
        "site": s, "colatitude_deg": 90 + la,
        "el_18.6y_min": el.min(), "el_18.6y_max": el.max(),
        "el_window_min": ser.earth_el_deg.min(), "el_window_max": ser.earth_el_deg.max(),
        "series_minus_track_maxabs_deg": np.abs(resid).max(),
    })
pd.DataFrame(lib).set_index("site").round(3)

sub-Earth lat -6.86..6.86°, lon -8.05..8.08°


,colatitude_deg,el_18.6y_min,el_18.6y_max,el_window_min,el_window_max,series_minus_track_maxabs_deg
site,,,,,,
site01_connecting_ridge,0.513,-7.516,6.276,-7.5,6.23,0.006
site04_shackleton_rim,0.215,-7.310,6.422,-7.3,6.37,0.006


## 3. Horizon vs known terrain
(a) Shackleton's floor: the lowest LOLA point within 15 km of the pole. From the rim site, the horizon toward it should
be the far rim, ~20 km away, near 0° (looking across the bowl), not a nearby wall.
(b) For each site, where the highest horizon is set: azimuth, distance, height and lat/lon of that terrain.

In [4]:
FROM_STEREO = Transformer.from_crs(pgda.POLAR_STEREO, "+proj=longlat +R=1737400 +no_defs", always_xy=True)
with rasterio.open(lola.RAW / "LDEM_75S_120M.LBL") as ds:
    pwin = rasterio.windows.from_bounds(-15e3, -15e3, 15e3, 15e3, ds.transform).round_offsets().round_lengths()
    dn = ds.read(1, window=pwin)
    zp = np.where(dn == ds.nodata, np.nan, dn * ds.scales[0])
    rr, cc = np.unravel_index(np.nanargmin(zp), zp.shape)
    fx, fy = ds.window_transform(pwin) * (cc + 0.5, rr + 0.5)
floor_lon, floor_lat = FROM_STEREO.transform(fx, fy)
print(f"Shackleton floor (LOLA min within 15 km of the pole): {floor_lat:.3f}, {floor_lon % 360:.2f} E, {np.nanmin(zp):.0f} m")

known = []
for s in SITE_IDS:
    h = terrain.horizon(s)
    la, lo = np.radians(SITES[s])
    if s == "site04_shackleton_rim":
        fla, flo = np.radians(floor_lat), np.radians(floor_lon)
        az_f = np.degrees(np.arctan2(np.sin(flo - lo) * np.cos(fla),
                                     np.cos(la) * np.sin(fla) - np.sin(la) * np.cos(fla) * np.cos(flo - lo))) % 360
        k = round(az_f / terrain.AZ_STEP_DEG) % len(h["elevation_deg"])
        print(f"{s}: toward the floor (az {az_f:.1f}): horizon {h['elevation_deg'][k]:.2f} deg set "
              f"{h['distance_km'][k]:.1f} km away (near field: {bool(h['near_field'][k])})")
    k = int(np.argmax(h["elevation_deg"]))
    az = np.radians(k * terrain.AZ_STEP_DEG)
    th = h["distance_km"][k] * 1e3 / terrain.R_M
    plat = np.arcsin(np.sin(la) * np.cos(th) + np.cos(la) * np.sin(th) * np.cos(az))
    plon = lo + np.arctan2(np.sin(az) * np.sin(th) * np.cos(la), np.cos(th) - np.sin(la) * np.sin(plat))
    rel = np.abs(np.asarray(hz[s]["elevationDeg"]) - np.round(h["elevation_deg"], 2)).max()
    known.append({"site": s, "max_horizon_deg": h["elevation_deg"][k], "at_az_deg": k * terrain.AZ_STEP_DEG,
                  "distance_km": h["distance_km"][k], "lat": np.degrees(plat), "lon_e": np.degrees(plon) % 360,
                  "min_horizon_deg": h["elevation_deg"].min(), "public_file_vs_rerun_maxabs": rel})
pd.DataFrame(known).set_index("site").round(2)

Shackleton floor (LOLA min within 15 km of the pole): -89.624, 118.59 E, -2866 m


site04_shackleton_rim: toward the floor (az 243.7): horizon -1.86 deg set 21.2 km away (near field: False)


,max_horizon_deg,at_az_deg,distance_km,lat,lon_e,min_horizon_deg,public_file_vs_rerun_maxabs
site,,,,,,,
site01_connecting_ridge,10.12,307.0,0.07,-89.49,223.49,-2.57,0.0
site04_shackleton_rim,1.02,30.5,9.93,-89.48,222.52,-2.66,0.0


## 4. Frame: Horizons (IAU Moon) vs the DEMs (MOON_ME, DE421)
The IAU rotation equations are quoted as precise to about 150 m on the surface (WGCCRE, via NAIF's spice_discussion,
May 2015). Treat that as a site-position error: move each site 150 m north/east/south/west, re-ray-cast the horizon,
and see how much the window DTE % (Earth centre above terrain, 0° mask) moves. The direction error it adds to Earth
(150 m / 1737 km ≈ 0.005°) is far below the horizon uncertainty, so position is what matters.

In [5]:
SHIFT_M = 150.0
frame = []
saved = dict(SITES)
try:
    for s in SITE_IDS:
        base = (win[s].earth_el_deg > horizon_at(s, win[s].earth_az_deg)).mean() * 100
        la0, lo0 = saved[s]
        for name, dn, de in [("N", 1, 0), ("E", 0, 1), ("S", -1, 0), ("W", 0, -1)]:
            # small-offset step on the sphere; at -89.5° one degree of longitude is only ~150 m, so compute it
            dlat = np.degrees(dn * SHIFT_M / terrain.R_M)
            dlon = np.degrees(de * SHIFT_M / (terrain.R_M * np.cos(np.radians(la0))))
            SITES[s] = (la0 + dlat, lo0 + dlon)
            h = terrain.horizon(s)["elevation_deg"]
            h_at_earth = np.interp(win[s].earth_az_deg % 360, np.arange(0, 360.5, 0.5), np.append(h, h[0]))
            frame.append({"site": s, "shift": name, "dte_window_pct": (win[s].earth_el_deg > h_at_earth).mean() * 100,
                          "base_pct": base, "max_horizon_change_deg": np.abs(h - np.asarray(hz[s]["elevationDeg"])).max()})
            SITES[s] = (la0, lo0)
finally:
    SITES.update(saved)
pd.DataFrame(frame).set_index(["site", "shift"]).round(2)

dte_window_pct  base_pct  \
site                    shift                             
site01_connecting_ridge N               46.93     46.87   
                        E               46.56     46.87   
                        S               46.79     46.87   
                        W               47.14     46.87   
site04_shackleton_rim   N               42.65     45.26   
                        E               44.87     45.26   
                        S               45.21     45.26   
                        W               27.17     45.26   

                               max_horizon_change_deg  
site                    shift                          
site01_connecting_ridge N                        2.30  
                        E                        3.41  
                        S                        4.26  
                        W                        4.44  
site04_shackleton_rim   N                        8.62  
                        E                       15.08  
                        S                        6.50  
                        W                       19.89